# Session 10 — Brain Networks and Graph Theory

**Goal of this session:** go from a set of regional signals to a network, and measure something about its shape.

*Python for Neuroscience, session 10 of 12.*

## Why this matters

No brain region does anything alone. The interesting question is usually not "is this region active" but "what is it coordinating with".

The standard move is: record several regions, correlate every pair, and treat the result as a network. Regions become nodes, strong correlations become edges, and a whole branch of mathematics becomes available to you.

## Nine regions, two systems

We simulate nine regions. Four of them share a hidden common driver, four share a different one, and the ninth listens to both. Each also has its own private noise.

Nothing here is a real brain. It is a network with a known answer, which is exactly what you want when you are learning to detect structure.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
n_timepoints = 300

regions = ["V1", "V2", "V4", "MT", "M1", "S1", "PMC", "SMA", "PCC"]
module = [0, 0, 0, 0, 1, 1, 1, 1, 2]      # 0 = visual, 1 = motor, 2 = the connector
coupling = [0.85, 0.80, 0.70, 0.60, 0.85, 0.75, 0.65, 0.60, 0.0]

driver_visual = rng.standard_normal(n_timepoints)
driver_motor = rng.standard_normal(n_timepoints)

signals = []
for i in range(len(regions)):
    if module[i] == 0:
        shared = coupling[i] * driver_visual
    elif module[i] == 1:
        shared = coupling[i] * driver_motor
    else:
        shared = 0.65 * (driver_visual + driver_motor) / np.sqrt(2)
    signals.append(shared + 0.7 * rng.standard_normal(n_timepoints))

signals = np.array(signals)
print(signals.shape, "= regions x timepoints")

## The correlation matrix

`np.corrcoef` correlates every row with every other row and hands back a square matrix. Row 3, column 5 is the correlation between region 3 and region 5. The diagonal is all ones, because everything correlates perfectly with itself.

In [ ]:
conn = np.corrcoef(signals)
print(conn.shape)
print(np.round(conn[:4, :4], 2))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(conn, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(regions)))
ax.set_yticks(range(len(regions)))
ax.set_xticklabels(regions, rotation=45, fontsize=12)
ax.set_yticklabels(regions, fontsize=12)
ax.set_title("Functional connectivity matrix", fontsize=15)
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("correlation", fontsize=12)
plt.tight_layout()
plt.show()

The two red blocks along the diagonal are the two systems. That block structure is what people mean when they talk about modules or communities in brain networks, and here you can read it straight off the picture.

## From matrix to graph

A matrix of correlations is not yet a network. You have to decide which correlations count as connections, which means choosing a threshold.

That choice is a real scientific decision and there is no universally correct value. Set it too low and everything connects to everything. Too high and your network falls apart into pieces.

In [ ]:
import networkx as nx

threshold = 0.3
adjacency = (conn > threshold) & ~np.eye(len(regions), dtype=bool)

G = nx.from_numpy_array(adjacency.astype(int))
G = nx.relabel_nodes(G, dict(enumerate(regions)))

print(G.number_of_nodes(), "nodes")
print(G.number_of_edges(), "edges")
print("connected:", nx.is_connected(G))

## Two metrics

**Degree** is how many connections a node has. High-degree nodes are hubs, and hub damage tends to matter more than damage elsewhere.

**Clustering coefficient** asks whether a node's neighbours are also connected to each other. A value near 1 means the node sits inside a tight local cluster. A low value on a well-connected node is the signature of a bridge between systems.

In [ ]:
degree = dict(G.degree())
clustering = nx.clustering(G)

print(f"{'region':8s} {'degree':>7s} {'clustering':>12s}")
for r in regions:
    print(f"{r:8s} {degree[r]:7d} {clustering[r]:12.2f}")

PCC has the most connections and a low clustering coefficient. It talks to both systems, and the systems do not talk to each other, so its neighbours are not neighbours of one another.

That combination, many connections and low clustering, is how graph theory identifies a connector hub. We built it in deliberately, and the metric found it.

## Drawing the network

In [ ]:
colours = {0: "#2b6cb0", 1: "#dd6b20", 2: "#38a169"}
node_colours = [colours[module[i]] for i in range(len(regions))]
node_sizes = [degree[r] * 320 + 300 for r in regions]

fig, ax = plt.subplots(figsize=(8.5, 7))
pos = nx.spring_layout(G, seed=1)
nx.draw_networkx_edges(G, pos, alpha=0.4, width=1.5, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colours, node_size=node_sizes,
                       alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=12, font_color="white",
                        font_weight="bold", ax=ax)
ax.set_title("Network at threshold r > 0.3\n"
             "node size = degree, colour = system", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

## Try it yourself

Change `threshold` to 0.15 and then to 0.5, redrawing each time. At 0.5 the network breaks into disconnected pieces and `nx.is_connected` returns False, which means several standard metrics stop being defined.

Any paper reporting graph metrics should tell you its threshold. If it does not, that is worth noticing.

**Next session:** real fMRI data, downloaded live.